# Bonsai Stage 1D: real pilot workload, GPU-batched (37 trajectories, all real constructions)

**One question**: how fast does the GPU run the *actual* Stage 1D pilot workload -- not a fixed-topology stand-in, but the real 37 trajectories Claude Code is running on the M1 right now: 10 lattice trajectories + 3 stochastic-control families x 3 graph realizations x 3 trajectory seeds (27), each construction requiring its own graph, not just its own initial condition.

**This uses the actual, already-verified project code**, not approximations: `build_class_topology`, `get_degree_stratified_nodes`, `degree_preserving_rewire`, `generate_matched_sparsity_topology`, `generate_historical_matched_sparsity_random` + `rescale_to_common_budget` are inlined below exactly as they exist in `src/bonsai/dynamics/` and `experiments/stage1b2_structured_transformation/stage1b2_core.py` on the `stage1d-lattice-and-pilot` branch. `PILOT_REALIZATION_SEEDS = [0, 1, 2]` matches `build_stage1d_constructions.py` exactly. The lattice construction is loaded directly from the already byte-exact-verified cached artifact rather than reimplemented (its full source wasn't re-read before writing this notebook -- loading the verified array is safer than a guess).

**Two files needed on Drive**, both in `/My Drive/bonsai/`: `class0_T.npy` (already there from earlier) and `class0_lattice.npy` (provided alongside this notebook).

**Fresh, self-contained notebook** -- does not depend on either earlier notebook's session state.

**Relies on the earlier prototype notebook's scipy/JAX verification** (not re-checked here) -- the RHS and solver code below is unchanged from what already passed that check.

**One caveat on the M1 comparison number**: Claude Code is currently re-running the real 37-trajectory pilot on the M1 (a NaN-handling fix for a degenerate isolated-node realization), so the ~61-minute figure used below is from the *first* run, not the corrected one -- close enough for a speed comparison, but worth updating once the corrected numbers land.


## Setup

**Run this cell, then Runtime > Restart session, then run everything below from the top.**


In [1]:
import os
import requests

# Fetch current session details from the internal Jupyter API
jupyter_ip = os.environ.get('COLAB_JUPYTER_IP', '172.28.0.12')
session_data = requests.get(f"http://{jupyter_ip}:9000/api/sessions").json()[0]

print("Session ID:", session_data['id'])
print("Notebook Name:", session_data['name'])

Session ID: 5a3f9dbb-6788-409b-9e52-59f772498d0a
Notebook Name: bonsai real pilot gpu benchmark-intellij-e5c56d47-9384-4ecc-9ae3-20ed92c6638d.ipynb


In [3]:
import os
import json
import requests
from pathlib import Path

# 1. Fetch current live PyCharm-Colab session details
jupyter_ip = os.environ.get('COLAB_JUPYTER_IP', '172.28.0.12')
session_data = requests.get(f"http://{jupyter_ip}:9000/api/sessions").json()[0]

session_id = session_data['id']
notebook_name = session_data['name']

# 2. Path to the colab-cli local registry
cli_config_path = Path.home() / ".config" / "colab-cli" / "sessions.json"
cli_config_path.parent.mkdir(parents=True, exist_ok=True)

# 3. Load existing or initialize empty sessions structure
if cli_config_path.exists() and cli_config_path.stat().st_size > 0:
    with open(cli_config_path, "r") as f:
        try:
            sessions_db = json.load(f)
        except json.JSONDecodeError:
            sessions_db = {}
else:
    sessions_db = {}

# 4. Inject PyCharm's backend tracking data into the schema
# (Adjust keys if your specific version uses a list instead of a dictionary mapping)
sessions_db[notebook_name] = {
    "session_id": session_id,
    "backend_ip": jupyter_ip,
    "hardware": "GPU/CPU (PyCharm Local Link)",
    "status": "active"
}

with open(cli_config_path, "w") as f:
    json.dump(sessions_db, f, indent=4)

print(f"Successfully registered '{notebook_name}' to ~/.config/colab-cli/sessions.json")


Successfully registered 'bonsai real pilot gpu benchmark-intellij-e5c56d47-9384-4ecc-9ae3-20ed92c6638d.ipynb' to ~/.config/colab-cli/sessions.json


In [1]:
!pip install -q -U diffrax "jax[cuda12]"

import os
import time
import urllib.request
import gzip
import struct
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)  # required for correctness -- do not remove
import jax.numpy as jnp
import diffrax
from scipy.integrate import solve_ivp

print('JAX devices:', jax.devices())
print('JAX default backend:', jax.default_backend())
print('JAX x64 enabled:', jax.config.jax_enable_x64)


JAX devices: [CudaDevice(id=0)]
JAX default backend: gpu
JAX x64 enabled: True


## Download raw KMNIST (train images + labels only, ~18MB) and load `T` / lattice from Drive

KMNIST downloaded directly from the official source (`codh.rois.ac.jp`) rather than uploaded manually -- only used to recompute `active_indices` / `ink_mask_active` (needed for the pilot constructions, not cached anywhere), NOT to rebuild `T` from scratch (T is loaded from the already byte-exact-verified `class0_T.npy`, then cross-checked against a from-scratch rebuild below).


In [2]:
def load_idx_images(path):
    opener = gzip.open if path.endswith('.gz') else open
    with opener(path, 'rb') as f:
        magic, n_images, n_rows, n_cols = struct.unpack('>IIII', f.read(16))
        assert magic == 2051, f'bad magic {magic}'
        data = np.frombuffer(f.read(), dtype=np.uint8)
        return data.reshape(n_images, n_rows, n_cols)

def load_idx_labels(path):
    opener = gzip.open if path.endswith('.gz') else open
    with opener(path, 'rb') as f:
        magic, n_labels = struct.unpack('>II', f.read(8))
        assert magic == 2049, f'bad magic {magic}'
        return np.frombuffer(f.read(), dtype=np.uint8)

KMNIST_BASE = 'http://codh.rois.ac.jp/kmnist/dataset/kmnist/'
for fname in ['train-images-idx3-ubyte.gz', 'train-labels-idx1-ubyte.gz']:
    if not os.path.exists(fname):
        urllib.request.urlretrieve(KMNIST_BASE + fname, fname)
        print(f'Downloaded {fname}')
    else:
        print(f'{fname} already present')

X_train = load_idx_images('train-images-idx3-ubyte.gz')
y_train = load_idx_labels('train-labels-idx1-ubyte.gz')
idx = np.where(y_train == 0)[0][:200]
images = X_train[idx].astype(np.float64) / 255.0
print(f'{len(images)} class-0 training images loaded, shape {images.shape}')


Downloaded train-images-idx3-ubyte.gz
Downloaded train-labels-idx1-ubyte.gz
200 class-0 training images loaded, shape (200, 28, 28)


In [6]:
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_T_PATH = '/content/drive/My Drive/bonsai/class0_T.npy'
# DRIVE_LATTICE_PATH = '/content/drive/My Drive/bonsai/class0_lattice.npy'
from google.colab import files
uploaded = files.upload()

DRIVE_T_PATH = '/content/class0_T.npy'
DRIVE_LATTICE_PATH = '/content/class0_lattice.npy'
assert os.path.exists(DRIVE_T_PATH), f'{DRIVE_T_PATH} not found -- fix the path before continuing'
assert os.path.exists(DRIVE_LATTICE_PATH), f'{DRIVE_LATTICE_PATH} not found -- upload it alongside class0_T.npy'
T_cached = np.load(DRIVE_T_PATH).astype(np.float64)
W_lattice = np.load(DRIVE_LATTICE_PATH).astype(np.float64)
print(f'Loaded cached T: shape {T_cached.shape}')
print(f'Loaded cached lattice: shape {W_lattice.shape}, total weight {W_lattice.sum():.6f}')


KeyboardInterrupt: 

## Reconstruct `active_indices` / `ink_mask_active` (exact port of `learned_topology_construction.py`)

Verifies the rebuilt T matches the cached `class0_T.npy` byte-exact before trusting `active_indices` for anything downstream -- same standard as the rest of this project.


In [ ]:
assert (T_cached is not None)
assert (W_lattice is not None)

H, W_dim = 28, 28

def _local_converged_phases(image, steps=150, dt=0.1, k_coupling=1.0, k_bias=1.0,
                             perturbation_std=0.01, seed=0):
    target_phase = image * np.pi
    rng = np.random.default_rng(seed)
    phases = (target_phase + rng.normal(0, perturbation_std, target_phase.shape)) % (2 * np.pi)
    for _ in range(steps):
        coupling = np.zeros_like(phases)
        coupling[1:, :] += np.sin(phases[:-1, :] - phases[1:, :])
        coupling[:-1, :] += np.sin(phases[1:, :] - phases[:-1, :])
        coupling[:, 1:] += np.sin(phases[:, :-1] - phases[:, 1:])
        coupling[:, :-1] += np.sin(phases[:, 1:] - phases[:, :-1])
        bias = np.sin(target_phase - phases)
        dtheta = k_coupling * coupling + k_bias * bias
        phases = (phases + dt * dtheta) % (2 * np.pi)
    return phases

def population_developmental_stat(images, steps=150, dt=0.1, k_coupling=1.0, k_bias=1.0):
    n_pixels = H * W_dim
    accum = np.zeros((n_pixels, n_pixels))
    for image in images:
        phases = _local_converged_phases(image, steps=steps, dt=dt, k_coupling=k_coupling, k_bias=k_bias).flatten()
        diff = phases[:, None] - phases[None, :]
        accum += np.cos(diff)
    W_learned = accum / len(images)
    np.fill_diagonal(W_learned, 0)
    return W_learned

def build_class_topology(images, prune_threshold=0.9, ink_threshold=0.15, **stat_kwargs):
    W_learned = population_developmental_stat(images, **stat_kwargs)
    mean_intensity = images.mean(axis=0).flatten()
    ink_mask = mean_intensity > ink_threshold
    background_pair_mask = np.outer(~ink_mask, ~ink_mask)
    pruned = np.where(np.abs(W_learned) > prune_threshold, W_learned, 0.0)
    pruned[background_pair_mask] = 0.0
    active_indices = np.where(np.any(pruned != 0, axis=1))[0]
    W_active = pruned[np.ix_(active_indices, active_indices)]
    return active_indices, W_active

print('Building T from scratch (200 images x 150-step local convergence -- this is the slow part, a few minutes on CPU)...')
t0 = time.perf_counter()
active_indices, T_rebuilt = build_class_topology(images)
print(f'Done in {time.perf_counter() - t0:.1f}s')

max_diff = np.max(np.abs(T_rebuilt - T_cached))
print(f'Max abs diff vs cached T: {max_diff:.3e}')
assert max_diff < 1e-9, 'rebuilt T does not match cached class0_T.npy -- STOP, something is wrong upstream'
print('PASS: rebuilt T matches cached T byte-exact. active_indices is trustworthy.')

W = T_cached  # use the already-cached, verified T from here on
n = W.shape[0]
mean_intensity = images.mean(axis=0).flatten()
ink_mask_active = (mean_intensity > 0.15)[active_indices]
print(f'ink_mask_active: {ink_mask_active.sum()} of {len(ink_mask_active)} active nodes are ink')


## Degree-stratified nodes (exact `get_degree_stratified_nodes` from `stage1b2_core.py`)


In [ ]:
def get_degree_stratified_nodes(W):
    degree = W.sum(axis=1)
    order = np.argsort(degree)
    n = len(order)
    return {'low': int(order[n // 10]), 'median': int(order[n // 2]), 'high': int(order[-n // 10])}

nodes_T = get_degree_stratified_nodes(W)
nodes_batch = [nodes_T['low'], nodes_T['median'], nodes_T['high']]
print('nodes_T:', nodes_T)


## Build the 9 stochastic-control pilot realizations (exact real construction functions)

3 families x 3 realization seeds. Lattice was already loaded above from the verified cached array.


In [ ]:
def degree_preserving_rewire(topology, ink_mask, n_swaps_multiplier=10, seed=0):
    N = topology.shape[0]
    triu_i, triu_j = np.triu_indices(N, k=1)
    weights = topology[triu_i, triu_j]
    nonzero_mask = weights != 0
    edges_i = list(triu_i[nonzero_mask]); edges_j = list(triu_j[nonzero_mask]); edge_weights = list(weights[nonzero_mask])
    n_edges = len(edges_i)
    rng = np.random.default_rng(seed)
    edge_set = {(min(i, j), max(i, j)) for i, j in zip(edges_i, edges_j)}
    n_swaps_target = n_swaps_multiplier * n_edges
    successful_swaps = 0; attempts = 0; max_attempts = n_swaps_target * 20
    while successful_swaps < n_swaps_target and attempts < max_attempts:
        attempts += 1
        idx1, idx2 = rng.choice(n_edges, size=2, replace=False)
        a, b = edges_i[idx1], edges_j[idx1]; c, d = edges_i[idx2], edges_j[idx2]
        if len({a, b, c, d}) < 4: continue
        new_edge1 = (min(a, d), max(a, d)); new_edge2 = (min(c, b), max(c, b))
        if new_edge1 == new_edge2 or new_edge1 in edge_set or new_edge2 in edge_set: continue
        i1, j1 = new_edge1; i2, j2 = new_edge2
        if (~ink_mask[i1] and ~ink_mask[j1]) or (~ink_mask[i2] and ~ink_mask[j2]): continue
        old_edge1 = (min(a, b), max(a, b)); old_edge2 = (min(c, d), max(c, d))
        edge_set.discard(old_edge1); edge_set.discard(old_edge2)
        edge_set.add(new_edge1); edge_set.add(new_edge2)
        edges_i[idx1], edges_j[idx1] = new_edge1
        edges_i[idx2], edges_j[idx2] = new_edge2
        successful_swaps += 1
    rewired = np.zeros((N, N))
    for i, j, w in zip(edges_i, edges_j, edge_weights):
        rewired[i, j] = w; rewired[j, i] = w
    return rewired

def generate_matched_sparsity_topology(real_topology, ink_mask, seed):
    N = real_topology.shape[0]
    rng = np.random.default_rng(seed)
    triu_i, triu_j = np.triu_indices(N, k=1)
    eligible = ~(~ink_mask[triu_i] & ~ink_mask[triu_j])
    eligible_i, eligible_j = triu_i[eligible], triu_j[eligible]
    real_values = real_topology[triu_i, triu_j]
    nonzero_mask = real_values != 0
    n_edges = nonzero_mask.sum()
    values_to_place = real_values[nonzero_mask].copy()
    rng.shuffle(values_to_place)
    chosen = rng.choice(len(eligible_i), size=n_edges, replace=False)
    chosen_i, chosen_j = eligible_i[chosen], eligible_j[chosen]
    random_topo = np.zeros((N, N))
    random_topo[chosen_i, chosen_j] = values_to_place
    random_topo[chosen_j, chosen_i] = values_to_place
    return random_topo

def generate_historical_matched_sparsity_random(real_topology, ink_mask, seed, n_edges=None):
    N = real_topology.shape[0]
    rng = np.random.default_rng(seed)
    triu_i, triu_j = np.triu_indices(N, k=1)
    eligible = ~(~ink_mask[triu_i] & ~ink_mask[triu_j])
    eligible_i, eligible_j = triu_i[eligible], triu_j[eligible]
    real_values = real_topology[triu_i, triu_j]
    nonzero_mask = real_values != 0
    values_pool = real_values[nonzero_mask].copy()
    if n_edges is None:
        n_edges = round(int(nonzero_mask.sum()) / 2)
    rng.shuffle(values_pool)
    values_to_place = values_pool[:n_edges]
    chosen = rng.choice(len(eligible_i), size=n_edges, replace=False)
    chosen_i, chosen_j = eligible_i[chosen], eligible_j[chosen]
    random_topo = np.zeros((N, N))
    random_topo[chosen_i, chosen_j] = values_to_place
    random_topo[chosen_j, chosen_i] = values_to_place
    return random_topo

def rescale_to_common_budget(A, target_mean_weighted_degree):
    mean_weighted_degree = A.sum(axis=1).mean()
    return A * (target_mean_weighted_degree / mean_weighted_degree)

PILOT_REALIZATION_SEEDS = [0, 1, 2]

pilot_constructions = {}
for family in ['rewired', 'hist_random', 'curr_random']:
    pilot_constructions[family] = {}
    for seed in PILOT_REALIZATION_SEEDS:
        if family == 'rewired':
            G = degree_preserving_rewire(W, ink_mask_active, seed=seed)
        elif family == 'hist_random':
            target = W.sum(axis=1).mean()
            raw = generate_historical_matched_sparsity_random(W, ink_mask_active, seed=seed)
            G = rescale_to_common_budget(raw, target)
        elif family == 'curr_random':
            G = generate_matched_sparsity_topology(W, ink_mask_active, seed=seed)
        pilot_constructions[family][seed] = G
        print(f'{family} seed={seed}: n_edges={np.count_nonzero(np.triu(G, 1))}, '
              f'mean_weighted_degree={G.sum(axis=1).mean():.6f}')


## RHS and single-trial solver (JAX/diffrax) -- unchanged from the verified prototype notebook


In [ ]:
K_COUPLING = 1.0
T_HORIZON = 2.5
RTOL, ATOL, MAX_STEP = 1e-6, 1e-8, 0.05
NEARBY_SCALE = 0.1
T_P_VALUES = [0, 0.833, 1.667, 2.5]
N_REPLICAS = 6
signs_batch = [1, -1]
amps_batch = [0.025, 0.2, 0.8]

def force_jacobian_jax(Wg, theta, k_coupling=K_COUPLING):
    diff = theta[None, :] - theta[:, None]
    J = k_coupling * Wg * jnp.cos(diff)
    J = J - jnp.diag(jnp.diag(J))
    J = J - jnp.diag(J.sum(axis=1))
    return J

def make_rhs_jax(Wg, k_coupling=K_COUPLING):
    n_local = Wg.shape[0]
    def rhs(t, y, args):
        theta = y[:n_local]
        delta = y[n_local:]
        diff = theta[None, :] - theta[:, None]
        dtheta = k_coupling * jnp.sum(Wg * jnp.sin(diff), axis=1)
        J = force_jacobian_jax(Wg, theta, k_coupling)
        ddelta = J @ delta
        return jnp.concatenate([dtheta, ddelta])
    return rhs

def run_one_trial_jax(Wg, theta0, node, sign, amplitude, k_coupling=K_COUPLING):
    n_local = theta0.shape[0]
    P = jnp.eye(n_local) - jnp.ones((n_local, n_local)) / n_local
    epsilon = sign * amplitude
    delta0 = jnp.zeros(n_local).at[node].set(1.0)
    delta0 = P @ delta0
    delta0 = delta0 / jnp.linalg.norm(delta0)
    y0 = jnp.concatenate([theta0, delta0])
    term = diffrax.ODETerm(make_rhs_jax(Wg, k_coupling))
    solver = diffrax.Tsit5()
    stepsize_controller = diffrax.PIDController(rtol=RTOL, atol=ATOL, dtmax=MAX_STEP)
    sol = diffrax.diffeqsolve(term, solver, t0=0.0, t1=T_HORIZON, dt0=0.01, y0=y0,
                              stepsize_controller=stepsize_controller, max_steps=200_000)
    y_final = sol.ys[-1]
    return y_final[:n_local], y_final[n_local:]

def rhs_theta_only_np(t, theta, Wg):
    diff = theta[None, :] - theta[:, None]
    return K_COUPLING * np.sum(Wg * np.sin(diff), axis=1)

def build_432_batch_for_graph(Wg, baseline_seeds):
    """Builds the (theta0, node, sign, amp) batch for one graph, across the given baseline seeds."""
    theta0_list, nodes_list, signs_list, amps_list = [], [], [], []
    for baseline_seed in baseline_seeds:
        replica_direction_seed = baseline_seed + 1
        rng_b = np.random.default_rng(baseline_seed)
        theta0_b = rng_b.uniform(0, 2 * np.pi, n)
        sol_b = solve_ivp(rhs_theta_only_np, (0, T_HORIZON), theta0_b, args=(Wg,), method='RK45',
                           rtol=RTOL, atol=ATOL, max_step=MAX_STEP, dense_output=True)
        states_at_tp = [sol_b.sol(tp) for tp in T_P_VALUES]
        rng_r = np.random.default_rng(replica_direction_seed)
        directions = [rng_r.uniform(-1, 1, n) for _ in range(N_REPLICAS)]
        theta0_states = []
        for state_at_tp in states_at_tp:
            for direction in directions:
                theta0_states.append(np.mod(state_at_tp + NEARBY_SCALE * direction, 2 * np.pi))
        for theta0_state in theta0_states:
            for node in nodes_batch:
                for sign in signs_batch:
                    for amp in amps_batch:
                        theta0_list.append(theta0_state)
                        nodes_list.append(node)
                        signs_list.append(sign)
                        amps_list.append(amp)
    return (jnp.asarray(np.stack(theta0_list)), jnp.asarray(nodes_list),
            jnp.asarray(signs_list, dtype=jnp.float64), jnp.asarray(amps_list, dtype=jnp.float64))

print('RHS, solver, and batch-builder defined.')


## Run all 37 trajectory-batches, timed, compared to the real M1 pilot

Loops over the 10 distinct graphs (not one giant vmap across all 37 -- naively repeating each graph across its full trial batch would need ~33GB, since 37 graphs x 432 trials each x 505x505x8 bytes adds up fast). Each graph's own trajectory-seed set gets one `vmap` call over its 432-trial batch, mirroring how the real pilot itself processes one trajectory (and therefore one graph) at a time.


In [ ]:
LATTICE_SEEDS = [3000, 3010, 3020, 3030, 3040, 3050, 3060, 3070, 3080, 3090]
PILOT_TRAJECTORY_SEEDS = [3000, 3010, 3020]  # first 3 of Stage 1C's, per DESIGN.md

batched_run = jax.jit(jax.vmap(run_one_trial_jax, in_axes=(None, 0, 0, 0, 0)))

jobs = [('lattice', None, jnp.asarray(W_lattice), LATTICE_SEEDS)]
for family, realizations in pilot_constructions.items():
    for seed, Wg in realizations.items():
        jobs.append((family, seed, jnp.asarray(Wg), PILOT_TRAJECTORY_SEEDS))

print(f'{len(jobs)} graphs to run ({sum(len(j[3]) for j in jobs)} trajectories total, should be 37)')

total_elapsed = 0.0
for name, seed, Wg_jax, baseline_seeds in jobs:
    theta0_b, nodes_b, signs_b, amps_b = build_432_batch_for_graph(np.asarray(Wg_jax), baseline_seeds)
    _ = batched_run(Wg_jax, theta0_b, nodes_b, signs_b, amps_b)
    jax.block_until_ready(_)  # warm-up / compile, excluded from timing
    t0 = time.perf_counter()
    result = batched_run(Wg_jax, theta0_b, nodes_b, signs_b, amps_b)
    jax.block_until_ready(result)
    elapsed = time.perf_counter() - t0
    total_elapsed += elapsed
    label = name if seed is None else f'{name} seed={seed}'
    print(f'{label}: {len(baseline_seeds)} trajectories, {elapsed:.2f}s')

M1_MINUTES_37_TRAJECTORIES = 61  # Claude Code's first-run figure; re-check once the corrected run lands
m1_seconds = M1_MINUTES_37_TRAJECTORIES * 60

print()
print(f'Total JAX GPU time, all 37 trajectories ({jax.default_backend()}): {total_elapsed:.2f}s')
print(f'M1 Mac Studio, real measured baseline: {m1_seconds}s ({M1_MINUTES_37_TRAJECTORIES} min, '
      f'first-run figure, 9-worker multiprocessing.Pool)')
print()
print(f'Speedup vs. real M1 baseline, full real pilot workload: {m1_seconds / total_elapsed:.1f}x')


## Honest caveats

- **The M1 comparison number is from the first pilot run**, before Claude Code's NaN/degenerate-node fix and re-run -- close enough for a speed comparison (the fix changes *analysis* correctness, not how long the underlying 37 trajectories take to compute), but worth swapping in the corrected figure once available.
- **This does not re-verify JAX against scipy in this notebook** -- relies on the earlier prototype notebook's verification, since the RHS/solver code is unchanged.
- **Fixed-coordinate intervention only** (`nodes_T`'s low/median/high, shared across every construction) -- matches DESIGN.md's primary protocol; role-matched intervention (each construction's own degree-stratified nodes) is a separate, secondary analysis not attempted here.
- **The isolated-node degeneracy Claude Code found** (hist_random seed=2, two of the three fixed-coordinate nodes landing on zero-degree positions in that realization) is a scientific-validity question for the *statistical* pilot analysis, not something this speed benchmark needs to handle specially -- the ODE still solves fine numerically on an isolated node (its tangent response is just trivially small), so it doesn't distort the timing.
